In [ ]:
!pip install streamlit pandas plotly wordcloud matplotlib transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 34.7 MB/s eta 0:00:00


In [ ]:
# Instalar cloudflared (solo una vez)
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!mv cloudflared /usr/local/bin/cloudflared

--2025-11-09 21:36:44--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2025.11.1/cloudflared-linux-amd64 [following]
--2025-11-09 21:36:45--  https://github.com/cloudflare/cloudflared/releases/download/2025.11.1/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/955e9d1b-ac5e-4188-8867-e5f53958a8fe?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-11-09T22%3A25%3A34Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-1

In [ ]:
%%writefile medical_chatbot_streamlit.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from collections import Counter
import re
import io
import torch
from transformers import pipeline

# ========== Configuración de la página ==========
st.set_page_config(page_title="MediBot - LLM Médico", page_icon="🩺", layout="wide")
st.title("🩺 MediBot: Visualización de Datos y Chatbot Médico")

# ========== Carga de datos ==========
@st.cache_data
def cargar_datos():
    try:
        df = pd.read_csv("/content/test.csv")
        if 'Conversation' not in df.columns:
            st.error("El archivo debe tener una columna llamada 'conversation'")
            return None
        return df
    except Exception as e:
        st.error(f"Error al cargar el archivo: {e}")
        return None

df = cargar_datos()
if df is None:
    st.stop()

# ========== Preprocesamiento ==========
@st.cache_data
def procesar_datos(_df):
    df_proc = _df.copy()
    df_proc["longitud"] = df_proc["Conversation"].apply(lambda x: len(str(x)))
    df_proc["palabras"] = df_proc["Conversation"].apply(lambda x: len(str(x).split()))
    df_proc["oraciones"] = df_proc["Conversation"].apply(lambda x: len(re.split(r'[.!?]+', str(x))) - 1)
    return df_proc

df_proc = procesar_datos(df)

# ========== Clasificador de temas (Llama 3 via HF) ==========
@st.cache_resource
def cargar_clasificador():
    try:
        classifier = pipeline(
            "text-classification",
            model="facebook/opt-350m",  # Modelo ligero para demo
            return_all_scores=True
        )
        return classifier
    except:
        return None

clasificador = cargar_clasificador()

# ========== Sidebar: Navegación ==========
st.sidebar.header("🧭 Navegación")
pagina = st.sidebar.radio(
    "Selecciona una sección:",
    ["🤖 Chatbot Médico", "📊 Vista de Datos", "📈 Estadísticas", "☁ Nube de Palabras", "🏷 Clasificación de Temas"],
    index=0
)

# ========== PÁGINA 1: CHATBOT ==========
if pagina == "🤖 Chatbot Médico":
    st.subheader("💬 Chatbot Médico (Demo)")

    # Simulación de respuestas
    respuestas = {
        "dolor de cabeza": "Podría ser migraña, tensión o deshidratación. Bebe agua, descansa y evita pantallas. Si persiste >3 días, consulta a un médico.",
        "fiebre": "Toma paracetamol cada 6-8h. Hidrátate bien. Si >38.5°C por más de 48h, acude al médico.",
        "tos": "Puede ser viral. Usa miel con limón, humidificador. Si hay flemas verdes o dificultad para respirar, ve al médico.",
        "default": "No soy un médico real. Te recomiendo consultar a un profesional de la salud para un diagnóstico preciso."
    }

    if "messages" not in st.session_state:
        st.session_state.messages = [{"role": "assistant", "content": "¡Hola! Soy MediBot. ¿En qué te puedo ayudar hoy?"}]

    # Mostrar historial
    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]):
            st.write(msg["content"])

    # Input del usuario
    if prompt := st.chat_input("Escribe tu consulta médica..."):
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"):
            st.write(prompt)

        # Respuesta simulada
        respuesta = respuestas["default"]
        for clave in respuestas:
            if clave in prompt.lower():
                respuesta = respuestas[clave]
                break

        with st.chat_message("assistant"):
            st.write(respuesta)
        st.session_state.messages.append({"role": "assistant", "content": respuesta})

# ========== PÁGINA 2: VISTA DE DATOS ==========
elif pagina == "📊 Vista de Datos":
    st.subheader("📄 Ejemplos de Conversaciones")

    st.sidebar.caption("Controles: Muestra de datos")
    n_mostrar = st.sidebar.slider("Número de ejemplos", 1, 20, 5)
    muestra = df["Conversation"].sample(n_mostrar, random_state=42)

    for i, texto in enumerate(muestra):
        with st.expander(f"Conversación {i+1} ({len(texto)} caracteres)"):
            st.write(texto)

    st.caption(f"Total de conversaciones: {len(df):,}")

# ========== PÁGINA 3: ESTADÍSTICAS ==========
elif pagina == "📈 Estadísticas":
    st.subheader("📊 Estadísticas del Dataset")

    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Total Conversaciones", f"{len(df):,}")
    col2.metric("Promedio Longitud", f"{df_proc['longitud'].mean():.0f} chars")
    col3.metric("Promedio Palabras", f"{df_proc['palabras'].mean():.1f}")
    col4.metric("Promedio Oraciones", f"{df_proc['oraciones'].mean():.1f}")

    # Gráficos
    c1, c2 = st.columns(2)
    with c1:
        fig1 = px.histogram(df_proc, x="longitud", nbins=50, title="Distribución de Longitud (caracteres)")
        st.plotly_chart(fig1, use_container_width=True)
    with c2:
        fig2 = px.histogram(df_proc, x="palabras", nbins=50, title="Distribución de Palabras")
        st.plotly_chart(fig2, use_container_width=True)

# ========== PÁGINA 4: NUBE DE PALABRAS ==========
elif pagina == "☁ Nube de Palabras":
    st.subheader("☁ Nube de Palabras Comunes")

    st.sidebar.caption("Controles: Nube de palabras")
    n_palabras = st.sidebar.slider("Máximo de palabras", 50, 300, 100)

    # Unir todo el texto
    texto_completo = " ".join(df["Conversation"].astype(str))
    palabras = re.findall(r'\b[a-zA-Z]+\b', texto_completo.lower())
    comunes = Counter(palabras).most_common(n_palabras)

    wc = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(dict(comunes))

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis("off")
    st.pyplot(fig)

# ========== PÁGINA 5: CLASIFICACIÓN DE TEMAS ==========
elif pagina == "🏷 Clasificación de Temas":
    st.subheader("🏷 Clasificación Automática de Temas (Demo con Llama-like)")

    if clasificador is None:
        st.warning("No se pudo cargar el modelo de clasificación. Usando simulación.")
        temas = ["Síntomas", "Diagnóstico", "Tratamiento", "Prevención"]
        muestra = df["Conversation"].sample(5)
        for i, conv in enumerate(muestra):
            tema = np.random.choice(temas)
            prob = np.random.uniform(0.6, 0.95)
            with st.expander(f"Conversación {i+1}"):
                st.write(conv[:300] + "...")
                st.progress(prob)
                st.caption(f"Tema predicho: *{tema}* ({prob:.1%})")
    else:
        st.info("Clasificando 5 ejemplos aleatorios...")
        muestra = df["Conversation"].sample(5)
        for conv in muestra:
            try:
                resultado = clasificador(conv[:500])[0]  # Limitar longitud
                label = resultado['label']
                score = resultado['score']
            except:
                label, score = "Desconocido", 0.5

            with st.expander(f"Ejemplo"):
                st.write(conv[:300] + "...")
                st.progress(score)
                st.caption(f"Tema: *{label}* ({score:.1%})")

Writing medical_chatbot_streamlit.py


In [ ]:
def run():
  !streamlit run medical_chatbot_streamlit.py &>/content/logs.txt &    #streamlit run el proyecto en &>/content/logs.txt
  !cloudflared tunnel --url http://localhost:8501 >/content/hola.log 2>&1&    #Tercero crea archivo url
  import time
  time.sleep(5)     #Se necesita unos segundo para que se cree el archivo hola.log
  with open("/content/hola.log", encoding="utf-8", errors="ignore") as f:   #utf8 quiere decir que hay caracteres especiales para que no los tome como error
    for line in f:
      if "trycloudflare" in line and "|" in line:
        print(line)

def kill():     #Acabar la conexion de streamlit
  !pkill streamlit

In [ ]:
run()

In [11]:
kill()